In [1]:
import os
import gc
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np

# ── ROOT DATA FOLDER ──────────────────────────────────────────────────────────
DATA_DIR = Path("D:/Data Challenge")

SIM_MONTHLY_DIR = DATA_DIR / "sim_monthly"
SIM_DAILY_DIR   = DATA_DIR / "sim_daily"
COSTS_FILE      = DATA_DIR / "costs" / "costs.parquet"
PRICES_FILE     = DATA_DIR / "prices" / "prices.parquet"

YEARS = [2020, 2021, 2022, 2023]
MAX_MONTH = "2023-12"
UNIVERSE_MODE = "intersection"   # "intersection" | "sim_monthly" | "union"

KEYS = ["EID", "MONTH", "PEAKID"]
SCEN_KEYS = ["EID", "MONTH", "PEAKID", "SCENARIOID"]

def year_file(dataset: str, year: int) -> Path:
    if dataset == "sim_monthly":
        return SIM_MONTHLY_DIR / f"sim_monthly_{year}.parquet"
    elif dataset == "sim_daily":
        return SIM_DAILY_DIR / f"sim_daily_{year}.parquet"
    else:
        raise ValueError(f"Unknown dataset: {dataset}")

# ── OUTPUT / BUILD FOLDERS ────────────────────────────────────────────────────
PROJECT_DIR = DATA_DIR.parent
BUILD_DIR = PROJECT_DIR / "build_master_outputs"
TMP_DIR = BUILD_DIR / "duckdb_tmp"

BUILD_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

SIM_MONTHLY_AGG_FILE = BUILD_DIR / "sim_monthly_agg.parquet"
SIM_DAILY_AGG_FILE   = BUILD_DIR / "sim_daily_agg.parquet"
MASTER_FILE          = BUILD_DIR / "master.parquet"

# ── DUCKDB CONNECTION ─────────────────────────────────────────────────────────
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=8")
con.execute(f"SET temp_directory='{TMP_DIR.as_posix()}'")

SEP = "=" * 78

# ── SMALL HELPERS ─────────────────────────────────────────────────────────────
def print_shape(df: pd.DataFrame, name: str) -> None:
    print(f"  {name:<24}: {df.shape[0]:,} rows × {df.shape[1]} cols")

def mem_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / 1e6

def assert_exists(path: Path) -> None:
    assert path.exists(), f"Missing file: {path}"

def assert_unique_keys(df: pd.DataFrame, keys: list[str], name: str) -> None:
    dupes = df.duplicated(keys).sum()
    assert dupes == 0, f"{name} has {dupes:,} duplicate rows on keys {keys}"

def assert_no_null_keys(df: pd.DataFrame, keys: list[str], name: str) -> None:
    n_null = int(df[keys].isnull().sum().sum())
    assert n_null == 0, f"{name} has {n_null:,} null values in key columns {keys}"

# ── FILE CHECK ────────────────────────────────────────────────────────────────
print(f"{SEP}\n  FILE CHECK\n{SEP}")

all_files = (
    [year_file("sim_monthly", y) for y in YEARS] +
    [year_file("sim_daily", y) for y in YEARS] +
    [COSTS_FILE, PRICES_FILE]
)

for f in all_files:
    exists = f.exists()
    icon = "✅" if exists else "❌"
    size = f"{f.stat().st_size/1e9:.2f} GB" if exists else ""
    print(f"  {icon}  {f.name:<40} {size}")

for f in all_files:
    assert_exists(f)

print(f"\n✅ Cell 1 done.")
print(f"   DuckDB version : {duckdb.__version__}")
print(f"   Data dir       : {DATA_DIR}")
print(f"   Build dir      : {BUILD_DIR}")
print(f"   Temp dir       : {TMP_DIR}")
print(f"   Universe mode  : {UNIVERSE_MODE}")
print(f"   Max month      : {MAX_MONTH}")

  FILE CHECK
  ✅  sim_monthly_2020.parquet                 2.97 GB
  ✅  sim_monthly_2021.parquet                 3.49 GB
  ✅  sim_monthly_2022.parquet                 3.76 GB
  ✅  sim_monthly_2023.parquet                 1.55 GB
  ✅  sim_daily_2020.parquet                   1.07 GB
  ✅  sim_daily_2021.parquet                   1.26 GB
  ✅  sim_daily_2022.parquet                   1.43 GB
  ✅  sim_daily_2023.parquet                   1.56 GB
  ✅  costs.parquet                            0.00 GB
  ✅  prices.parquet                           0.00 GB

✅ Cell 1 done.
   DuckDB version : 1.5.0
   Data dir       : D:\Data Challenge
   Build dir      : D:\build_master_outputs
   Temp dir       : D:\build_master_outputs\duckdb_tmp
   Universe mode  : intersection
   Max month      : 2023-12


In [2]:
# Check row counts via DuckDB
costs_count = con.execute(f"SELECT count(*) FROM read_parquet('{COSTS_FILE.as_posix()}')").fetchone()[0]
prices_count = con.execute(f"SELECT count(*) FROM read_parquet('{PRICES_FILE.as_posix()}')").fetchone()[0]

print(f"Costs rows: {costs_count:,}")
print(f"Prices rows: {prices_count:,}")

# To see columns (the other half of 'shape')
print("Costs Columns:", con.execute(f"DESCRIBE SELECT * FROM read_parquet('{COSTS_FILE.as_posix()}')").df()['column_name'].tolist())

Costs rows: 9,092
Prices rows: 453,167
Costs Columns: ['EID', 'MONTH', 'PEAKID', 'C']


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — BUILD sim_monthly_agg
# HE-correct MONTH assignment + yearly aggregation + boundary deduplication
# Output grain: (EID, MONTH, PEAKID, SCENARIOID)
# ══════════════════════════════════════════════════════════════════════════════

print(f"{SEP}\n  CELL 2 — BUILD sim_monthly_agg\n{SEP}")

sim_feature_cols = """
    SUM(PSM)                       AS PSM_sum,
    AVG(ACTIVATIONLEVEL)           AS ACT_mean,
    AVG(WINDIMPACT)                AS WIND_mean,
    AVG(SOLARIMPACT)               AS SOLAR_mean,
    AVG(HYDROIMPACT)               AS HYDRO_mean,
    AVG(NONRENEWBALIMPACT)         AS NONRENWBAL_mean,
    AVG(EXTERNALIMPACT)            AS EXTERNAL_mean,
    AVG(LOADIMPACT)                AS LOAD_mean,
    AVG(TRANSMISSIONOUTAGEIMPACT)  AS TRANSOUTAGE_mean,
    COUNT(*)                       AS n_hours
"""

sim_monthly_parts = []

for year in YEARS:
    path = year_file("sim_monthly", year)
    print(f"  {year}...", end=" ", flush=True)

    part = con.execute(f"""
        SELECT
            {year} AS source_year,
            EID,
            STRFTIME(CAST(DATETIME AS TIMESTAMP) - INTERVAL 1 HOUR, '%Y-%m') AS MONTH,
            PEAKID,
            SCENARIOID,
            {sim_feature_cols}
        FROM read_parquet('{path.as_posix()}')
        WHERE STRFTIME(CAST(DATETIME AS TIMESTAMP) - INTERVAL 1 HOUR, '%Y-%m') <= '{MAX_MONTH}'
        GROUP BY EID, MONTH, PEAKID, SCENARIOID
    """).df()

    sim_monthly_parts.append(part)
    print(f"done ({len(part):,} rows)")

sim_monthly_agg = pd.concat(sim_monthly_parts, ignore_index=True)
del sim_monthly_parts
gc.collect()

sim_monthly_agg["year"] = sim_monthly_agg["MONTH"].str[:4].astype(int)
sim_monthly_agg["month_num"] = sim_monthly_agg["MONTH"].str[5:].astype(int)

assert_no_null_keys(sim_monthly_agg, SCEN_KEYS, "sim_monthly_agg_before_dedup")

# ── diagnose duplicates across yearly files ──────────────────────────────────
dupes_mask = sim_monthly_agg.duplicated(SCEN_KEYS, keep=False)
dupes_df = sim_monthly_agg.loc[dupes_mask].copy()

print_shape(sim_monthly_agg, "sim_monthly_agg before dedup")
print(f"  Memory                    : {mem_mb(sim_monthly_agg):.1f} MB")
print(f"  Duplicate rows on {SCEN_KEYS}: {len(dupes_df):,}")

if len(dupes_df) > 0:
    print(f"\n{'─'*70}\n  DUPLICATE DIAGNOSTICS — sim_monthly_agg\n{'─'*70}")

    print("  Duplicate rows by MONTH (top 20):")
    print(
        dupes_df.groupby("MONTH").size()
        .sort_values(ascending=False)
        .head(20)
        .to_string()
    )

    print("\n  Duplicate rows by source_year:")
    print(
        dupes_df.groupby("source_year").size()
        .sort_values(ascending=False)
        .to_string()
    )

    print("\n  First 20 duplicate keys:")
    print(
        dupes_df[["source_year"] + SCEN_KEYS]
        .sort_values(SCEN_KEYS + ["source_year"])
        .head(20)
        .to_string(index=False)
    )

    # Optional: check whether duplicates conflict in values
    value_cols = [
        "PSM_sum", "ACT_mean", "WIND_mean", "SOLAR_mean", "HYDRO_mean",
        "NONRENWBAL_mean", "EXTERNAL_mean", "LOAD_mean", "TRANSOUTAGE_mean", "n_hours"
    ]

    dup_conflicts = (
        dupes_df.groupby(SCEN_KEYS)[value_cols]
        .nunique()
        .reset_index()
    )
    conflicting = dup_conflicts[(dup_conflicts[value_cols] > 1).any(axis=1)]

    print(f"\n  Conflicting duplicate keys (different values across files): {len(conflicting):,}")

    print("\n  Deduplicating by keeping the row from the highest source_year...")
    sim_monthly_agg = (
        sim_monthly_agg
        .sort_values(["source_year"] + SCEN_KEYS)
        .drop_duplicates(subset=SCEN_KEYS, keep="last")
        .reset_index(drop=True)
    )

assert_unique_keys(sim_monthly_agg, SCEN_KEYS, "sim_monthly_agg_after_dedup")

print_shape(sim_monthly_agg, "sim_monthly_agg final")
print(f"  Memory                    : {mem_mb(sim_monthly_agg):.1f} MB")

sim_monthly_agg.to_parquet(SIM_MONTHLY_AGG_FILE, index=False)
print(f"\nSaved → {SIM_MONTHLY_AGG_FILE}")

print(f"\nSample (first 5 rows):")
print(sim_monthly_agg.head(5).to_string(index=False))

print(f"\n✅ Cell 2 done.")

  CELL 2 — BUILD sim_monthly_agg
  2020... 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done (333,024 rows)
  2021... 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done (393,228 rows)
  2022... 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done (449,328 rows)
  2023... 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done (495,630 rows)
  sim_monthly_agg before dedup: 1,671,210 rows × 17 cols
  Memory                    : 198.9 MB
  Duplicate rows on ['EID', 'MONTH', 'PEAKID', 'SCENARIOID']: 0
  sim_monthly_agg final   : 1,671,210 rows × 17 cols
  Memory                    : 198.9 MB

Saved → D:\build_master_outputs\sim_monthly_agg.parquet

Sample (first 5 rows):
 source_year  EID   MONTH  PEAKID  SCENARIOID  PSM_sum  ACT_mean  WIND_mean  SOLAR_mean  HYDRO_mean  NONRENWBAL_mean  EXTERNAL_mean  LOAD_mean  TRANSOUTAGE_mean  n_hours  year  month_num
        2020    4 2020-09       1           1      0.0  3.352456   2.523659   -0.029345   -0.130120         0.548733       0.108268   0.041313         -0.079378      336  2020          9
        2020    9 2020-01       0           1      0.0  0.774165   0.078274    0.000613   -0.052442         0.842328      -0.059628  -0.002466          0.029741      392  2020          1
        2020   17 2020-11       0           1      0.0 11.241444  12.933026    0.04065

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — BUILD sim_daily_agg
# HE-correct MONTH assignment + yearly aggregation + boundary deduplication
# Output grain: (EID, MONTH, PEAKID, SCENARIOID)
# ══════════════════════════════════════════════════════════════════════════════

print(f"{SEP}\n  CELL 3 — BUILD sim_daily_agg\n{SEP}")

daily_feature_cols = """
    AVG(PSD)                       AS PSD_mean,
    MAX(PSD)                       AS PSD_max,
    AVG(ACTIVATIONLEVEL)           AS ACTD_mean,
    AVG(WINDIMPACT)                AS WINDD_mean,
    AVG(SOLARIMPACT)               AS SOLARD_mean,
    AVG(HYDROIMPACT)               AS HYDROD_mean,
    AVG(NONRENEWBALIMPACT)         AS NONRENWBALD_mean,
    AVG(EXTERNALIMPACT)            AS EXTERNALD_mean,
    AVG(LOADIMPACT)                AS LOADD_mean,
    AVG(TRANSMISSIONOUTAGEIMPACT)  AS TRANSOUTAGED_mean,
    COUNT(*)                       AS n_hours_daily
"""

sim_daily_parts = []

for year in YEARS:
    path = year_file("sim_daily", year)
    print(f"  {year}...", end=" ", flush=True)

    part = con.execute(f"""
        SELECT
            {year} AS source_year,
            EID,
            STRFTIME(CAST(DATETIME AS TIMESTAMP) - INTERVAL 1 HOUR, '%Y-%m') AS MONTH,
            PEAKID,
            SCENARIOID,
            {daily_feature_cols}
        FROM read_parquet('{path.as_posix()}')
        WHERE STRFTIME(CAST(DATETIME AS TIMESTAMP) - INTERVAL 1 HOUR, '%Y-%m') <= '{MAX_MONTH}'
        GROUP BY EID, MONTH, PEAKID, SCENARIOID
    """).df()

    sim_daily_parts.append(part)
    print(f"done ({len(part):,} rows)")

sim_daily_agg = pd.concat(sim_daily_parts, ignore_index=True)
del sim_daily_parts
gc.collect()

sim_daily_agg["year"] = sim_daily_agg["MONTH"].str[:4].astype(int)
sim_daily_agg["month_num"] = sim_daily_agg["MONTH"].str[5:].astype(int)

assert_no_null_keys(sim_daily_agg, SCEN_KEYS, "sim_daily_agg_before_dedup")

# ── diagnose duplicates across yearly files ──────────────────────────────────
dupes_mask = sim_daily_agg.duplicated(SCEN_KEYS, keep=False)
dupes_df = sim_daily_agg.loc[dupes_mask].copy()

print_shape(sim_daily_agg, "sim_daily_agg before dedup")
print(f"  Memory                    : {mem_mb(sim_daily_agg):.1f} MB")
print(f"  Duplicate rows on {SCEN_KEYS}: {len(dupes_df):,}")

if len(dupes_df) > 0:
    print(f"\n{'─'*70}\n  DUPLICATE DIAGNOSTICS — sim_daily_agg\n{'─'*70}")

    print("  Duplicate rows by MONTH (top 20):")
    print(
        dupes_df.groupby("MONTH").size()
        .sort_values(ascending=False)
        .head(20)
        .to_string()
    )

    print("\n  Duplicate rows by source_year:")
    print(
        dupes_df.groupby("source_year").size()
        .sort_values(ascending=False)
        .to_string()
    )

    print("\n  First 20 duplicate keys:")
    print(
        dupes_df[["source_year"] + SCEN_KEYS]
        .sort_values(SCEN_KEYS + ["source_year"])
        .head(20)
        .to_string(index=False)
    )

    value_cols = [
        "PSD_mean", "PSD_max", "ACTD_mean", "WINDD_mean", "SOLARD_mean",
        "HYDROD_mean", "NONRENWBALD_mean", "EXTERNALD_mean",
        "LOADD_mean", "TRANSOUTAGED_mean", "n_hours_daily"
    ]

    dup_conflicts = (
        dupes_df.groupby(SCEN_KEYS)[value_cols]
        .nunique()
        .reset_index()
    )
    conflicting = dup_conflicts[(dup_conflicts[value_cols] > 1).any(axis=1)]

    print(f"\n  Conflicting duplicate keys (different values across files): {len(conflicting):,}")

    print("\n  Deduplicating by keeping the row from the highest source_year...")
    sim_daily_agg = (
        sim_daily_agg
        .sort_values(["source_year"] + SCEN_KEYS)
        .drop_duplicates(subset=SCEN_KEYS, keep="last")
        .reset_index(drop=True)
    )

assert_unique_keys(sim_daily_agg, SCEN_KEYS, "sim_daily_agg_after_dedup")

print_shape(sim_daily_agg, "sim_daily_agg final")
print(f"  Memory                    : {mem_mb(sim_daily_agg):.1f} MB")

sim_daily_agg.to_parquet(SIM_DAILY_AGG_FILE, index=False)
print(f"\nSaved → {SIM_DAILY_AGG_FILE}")

print(f"\nSample (first 5 rows):")
print(sim_daily_agg.head(5).to_string(index=False))

print(f"\n✅ Cell 3 done.")

  CELL 3 — BUILD sim_daily_agg
  2020... 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done (340,878 rows)
  2021... 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done (398,340 rows)
  2022... 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done (452,436 rows)
  2023... 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done (501,534 rows)
  sim_daily_agg before dedup: 1,693,188 rows × 18 cols
  Memory                    : 208.3 MB
  Duplicate rows on ['EID', 'MONTH', 'PEAKID', 'SCENARIOID']: 0
  sim_daily_agg final     : 1,693,188 rows × 18 cols
  Memory                    : 208.3 MB

Saved → D:\build_master_outputs\sim_daily_agg.parquet

Sample (first 5 rows):
 source_year  EID   MONTH  PEAKID  SCENARIOID  PSD_mean  PSD_max  ACTD_mean  WINDD_mean  SOLARD_mean  HYDROD_mean  NONRENWBALD_mean  EXTERNALD_mean  LOADD_mean  TRANSOUTAGED_mean  n_hours_daily  year  month_num
        2020  134 2020-09       1           1       0.0     -0.0   6.018478    1.750121    -0.055246     3.221552         -5.503904        2.433291    0.698957          -2.815669            480  2020          9
        2020  137 2020-09       1           1       0.0     -0.0  18.489581   -0.193961    -0.000191    -0.754351          1.491582       -0.186099   -0.332472          -8.597577            480  2020          9
        2020  138 

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — BUILD prices_agg + LOAD costs_agg
# prices: hourly -> monthly with HE correction
# costs : already monthly, just load / filter / validate
# ══════════════════════════════════════════════════════════════════════════════

print(f"{SEP}\n  CELL 4 — BUILD prices_agg + LOAD costs_agg\n{SEP}")

# ── prices_agg ────────────────────────────────────────────────────────────────
print("STEP A — aggregating prices with HE-correct MONTH...", flush=True)

prices_agg = con.execute(f"""
    SELECT
        EID,
        STRFTIME(CAST(DATETIME AS TIMESTAMP) - INTERVAL 1 HOUR, '%Y-%m') AS MONTH,
        PEAKID,
        SUM(PRICEREALIZED) AS PR
    FROM read_parquet('{PRICES_FILE.as_posix()}')
    WHERE STRFTIME(CAST(DATETIME AS TIMESTAMP) - INTERVAL 1 HOUR, '%Y-%m') <= '{MAX_MONTH}'
    GROUP BY EID, MONTH, PEAKID
""").df()

assert_no_null_keys(prices_agg, KEYS, "prices_agg")
assert_unique_keys(prices_agg, KEYS, "prices_agg")

print_shape(prices_agg, "prices_agg")
print(f"  Memory                    : {mem_mb(prices_agg):.1f} MB")

print("\nSample prices_agg (first 5 rows):")
print(prices_agg.head(5).to_string(index=False))

# ── costs_agg ────────────────────────────────────────────────────────────────
print("\nSTEP B — loading costs_agg...", flush=True)

costs_agg = pd.read_parquet(COSTS_FILE)
costs_agg = costs_agg[costs_agg["MONTH"] <= MAX_MONTH].copy()

assert_no_null_keys(costs_agg, KEYS, "costs_agg")
assert_unique_keys(costs_agg, KEYS, "costs_agg")

print_shape(costs_agg, "costs_agg")
print(f"  Memory                    : {mem_mb(costs_agg):.1f} MB")

print("\nSample costs_agg (first 5 rows):")
print(costs_agg.head(5).to_string(index=False))

print(f"\n✅ Cell 4 done.")

  CELL 4 — BUILD prices_agg + LOAD costs_agg
STEP A — aggregating prices with HE-correct MONTH...
  prices_agg              : 21,345 rows × 4 cols
  Memory                    : 0.6 MB

Sample prices_agg (first 5 rows):
 EID   MONTH  PEAKID          PR
   3 2020-05       1  -23.885000
   3 2020-06       1 -171.807201
   3 2021-08       0 -663.409700
  12 2021-06       1 -440.553102
 229 2021-03       1 -147.954897

STEP B — loading costs_agg...
  costs_agg               : 9,092 rows × 4 cols
  Memory                    : 0.2 MB

Sample costs_agg (first 5 rows):
 EID   MONTH  PEAKID         C
   3 2020-08       0 18.839701
   3 2020-09       0 15.829500
   3 2020-10       0 67.694901
   3 2020-10       1  1.245600
   3 2020-12       0 66.708801

✅ Cell 4 done.


In [10]:
# ======================================================================
# CELL 5 -- BUILD MASTER DATASET
#
# Joins sim_monthly_agg, sim_daily_agg, prices_agg, costs_agg onto a
# universe of (EID, MONTH, PEAKID) determined by UNIVERSE_MODE.
#
# Universe modes (set in Cell 1):
#   'intersection' -- only opps present in BOTH sim_monthly AND sim_daily
#   'union'        -- any opp present in either source
#   'sim_monthly'  -- sim_monthly alone (largest, no daily constraint)
#
# All joins are LEFT joins (zero-implicit convention):
#   PR = 0 when (EID, MONTH, PEAKID) absent from prices
#   C  = 0 when (EID, MONTH, PEAKID) absent from costs
#
# Sim feature NaNs are kept intentionally (indicate missing signal).
# ======================================================================

print(f"{SEP}\n  CELL 5 -- BUILD MASTER DATASET\n{SEP}")

# -- STEP 1: Universe -------------------------------------------------------
print(f"\nSTEP 1 -- building universe (mode: {UNIVERSE_MODE})...", flush=True)

u_monthly = sim_monthly_agg[KEYS].drop_duplicates()
u_daily   = sim_daily_agg[KEYS].drop_duplicates()

if UNIVERSE_MODE == "intersection":
    universe = u_monthly.merge(u_daily, on=KEYS, how="inner")
elif UNIVERSE_MODE == "union":
    universe = pd.concat([u_monthly, u_daily]).drop_duplicates().reset_index(drop=True)
else:  # 'sim_monthly'
    universe = u_monthly.copy()

universe = universe.sort_values(KEYS).reset_index(drop=True)

print(f"  sim_monthly unique keys   : {len(u_monthly):>10,}")
print(f"  sim_daily   unique keys   : {len(u_daily):>10,}")
print(f"  Universe ({UNIVERSE_MODE:<13})   : {len(universe):>10,}")
if UNIVERSE_MODE == "intersection":
    dropped = len(u_monthly) - len(universe)
    print(f"  Dropped (monthly-only)    : {dropped:>10,}  ({100*dropped/len(u_monthly):.1f}% of sim_monthly)")

del u_monthly, u_daily; gc.collect()

# -- STEP 2: Pivot sim_monthly cross-scenario features ----------------------
# sim_monthly_agg has 3 rows per (EID, MONTH, PEAKID) -- one per SCENARIOID.
# Pivot to 1 row per key with per-scenario columns + consensus features.

print(f"\nSTEP 2 -- pivoting sim_monthly features...", flush=True)

SIM_M_VARS = ["PSM_sum", "ACT_mean", "WIND_mean", "SOLAR_mean",
              "HYDRO_mean", "NONRENWBAL_mean", "EXTERNAL_mean",
              "LOAD_mean", "TRANSOUTAGE_mean"]

sim_m_pivot = (
    sim_monthly_agg
    .pivot_table(index=KEYS, columns="SCENARIOID", values=SIM_M_VARS)
    .round(6)
)
sim_m_pivot.columns = [f"{var}_S{int(s)}" for var, s in sim_m_pivot.columns]
sim_m_pivot = sim_m_pivot.reset_index()

for var in ["PSM_sum", "ACT_mean"]:
    s_cols = [f"{var}_S1", f"{var}_S2", f"{var}_S3"]
    sim_m_pivot[f"{var}_mean"] = sim_m_pivot[s_cols].mean(axis=1)
    sim_m_pivot[f"{var}_std"]  = sim_m_pivot[s_cols].std(axis=1)
sim_m_pivot["PSM_n_pos"] = (
    sim_m_pivot[["PSM_sum_S1", "PSM_sum_S2", "PSM_sum_S3"]] > 0
).sum(axis=1)

print_shape(sim_m_pivot, "sim_m_pivot")

# -- STEP 3: Pivot sim_daily cross-scenario features -------------------------

print(f"\nSTEP 3 -- pivoting sim_daily features...", flush=True)

SIM_D_VARS = ["PSD_mean", "PSD_max", "ACTD_mean"]

sim_d_pivot = (
    sim_daily_agg
    .pivot_table(index=KEYS, columns="SCENARIOID", values=SIM_D_VARS)
    .round(6)
)
sim_d_pivot.columns = [f"{var}_S{int(s)}" for var, s in sim_d_pivot.columns]
sim_d_pivot = sim_d_pivot.reset_index()

for var in SIM_D_VARS:
    s_cols = [f"{var}_S1", f"{var}_S2", f"{var}_S3"]
    sim_d_pivot[f"{var}_mean"] = sim_d_pivot[s_cols].mean(axis=1)
    sim_d_pivot[f"{var}_std"]  = sim_d_pivot[s_cols].std(axis=1)

print_shape(sim_d_pivot, "sim_d_pivot")

# -- STEP 4: Left-join everything onto universe ------------------------------

print(f"\nSTEP 4 -- joining all sources onto universe...", flush=True)

master = (
    universe
    .merge(sim_m_pivot, on=KEYS, how="left")
    .merge(sim_d_pivot, on=KEYS, how="left")
    .merge(prices_agg,  on=KEYS, how="left")
    .merge(costs_agg[KEYS + ["C"]], on=KEYS, how="left")
)

assert len(master) == len(universe), (
    f"Row count mismatch: {len(master):,} != {len(universe):,}  "
    f"-- check for duplicate keys in a pivot frame"
)

master["PR"] = master["PR"].fillna(0.0)  # zero-implicit
master["C"]  = master["C"].fillna(0.0)   # zero-implicit

del sim_m_pivot, sim_d_pivot; gc.collect()

# -- STEP 5: Labels + convenience columns ------------------------------------

master["profit"]     = master["PR"] - master["C"]
master["profitable"] = (master["profit"] > 0).astype(int)
master["year"]       = master["MONTH"].str[:4].astype(int)
master["month_num"]  = master["MONTH"].str[5:].astype(int)

assert_unique_keys(master, KEYS, "master")
assert_no_null_keys(master, KEYS, "master")

# -- Save -------------------------------------------------------------------

# -- Save -------------------------------------------------------------------

MASTER_CSV_FILE = MASTER_FILE.with_suffix(".csv")

master.to_parquet(MASTER_FILE, index=False)
print(f"\nSaved --> {MASTER_FILE}")

master.to_csv(MASTER_CSV_FILE, index=False)
print(f"Saved --> {MASTER_CSV_FILE}")
# -- Diagnostics ------------------------------------------------------------

n_total    = len(master)
n_prof     = int(master["profitable"].sum())
n_pr0_c0   = int(((master.PR == 0) & (master.C == 0)).sum())
n_pr0_cg0  = int(((master.PR == 0) & (master.C > 0)).sum())
n_prg0_c0  = int(((master.PR > 0)  & (master.C == 0)).sum())
n_prg0_cg0 = n_total - n_pr0_c0 - n_pr0_cg0 - n_prg0_c0

SEP2 = chr(9472) * 70
print(f"\n{SEP2}\n  MASTER OVERVIEW\n{SEP2}")
print(f"  Shape                      : {master.shape[0]:,} rows x {master.shape[1]} cols")
print(f"  Memory                     : {mem_mb(master):.1f} MB")
print(f"  Profitable (PR - C > 0)    : {n_prof:,}  ({100*n_prof/n_total:.2f}%)")
print(f"  Not profitable             : {n_total-n_prof:,}  ({100*(1-n_prof/n_total):.2f}%)")
print(f"\n  Breakdown by (PR, C) sign:")
print(f"    PR=0, C=0  : {n_pr0_c0:>10,}  (never activated, free)")
print(f"    PR=0, C>0  : {n_pr0_cg0:>10,}  (never activated, has cost)")
print(f"    PR>0, C=0  : {n_prg0_c0:>10,}  (earns freely)")
print(f"    PR>0, C>0  : {n_prg0_cg0:>10,}  (normal opportunities)")

pct = (master.groupby(["year", "PEAKID"])["profitable"]
             .mean().mul(100).unstack("PEAKID")
             .rename(columns={0: "OFF-Peak", 1: "ON-Peak"}))
print(f"\n  % profitable by year & PEAKID:")
print(pct.round(2).to_string())

print(f"\n  First 10 rows:")
print(master.head(10).to_string(index=False))

print(f"\nCell 5 done.  master.parquet --> {len(master):,} rows x {master.shape[1]} cols")


  CELL 5 -- BUILD MASTER DATASET

STEP 1 -- building universe (mode: intersection)...
  sim_monthly unique keys   :    557,070
  sim_daily   unique keys   :    564,396
  Universe (intersection )   :    556,630
  Dropped (monthly-only)    :        440  (0.1% of sim_monthly)

STEP 2 -- pivoting sim_monthly features...
  sim_m_pivot             : 557,070 rows × 35 cols

STEP 3 -- pivoting sim_daily features...
  sim_d_pivot             : 564,396 rows × 18 cols

STEP 4 -- joining all sources onto universe...

Saved --> D:\build_master_outputs\master.parquet
Saved --> D:\build_master_outputs\master.csv

──────────────────────────────────────────────────────────────────────
  MASTER OVERVIEW
──────────────────────────────────────────────────────────────────────
  Shape                      : 556,630 rows x 56 cols
  Memory                     : 232.7 MB
  Profitable (PR - C > 0)    : 3,242  (0.58%)
  Not profitable             : 553,388  (99.42%)

  Breakdown by (PR, C) sign:
    PR=0, C=0  